# Suite TRN — Translators

Every pair of dialects has its own bidirectional translator (`@mbse/expressions/Translators`), declared as rules
that pair two patterns with holes. Translating co-traverses each rule's pattern with the expression to bind the
holes, then instantiates the other side, so every term is translated once and sharing is kept. A dialect with no
binding gets a let's value substituted for its name, and imports are dropped where the other side cannot declare
them, or added where the translation needs them (`import numpy as np`). Translations keep an expression's form;
its value follows each dialect's rules, and this suite shows where those agree and where they do not. (Evaluating
the NumPy style with numpy is the Python implementation's only.)

In [ ]:
import { Domains as D, Evaluators as V, Expressions as E } from "@mbse/expressions";
import * as Ccpp from "@mbse/expressions/Dialects/Ccpp";
import * as Excel from "@mbse/expressions/Dialects/Excel";
import * as Latex from "@mbse/expressions/Dialects/Latex";
import * as Matlab from "@mbse/expressions/Dialects/Matlab";
import * as Python from "@mbse/expressions/Dialects/Python";
import * as SystemVerilog from "@mbse/expressions/Dialects/SystemVerilog";
import { Terms as F } from "@mbse/expressions/Framework";
import { Convert, Elide, Hole, Inline, Pairwise, Pattern, Prelude, Rule, holes, renames } from "@mbse/expressions/Framework/Translators";
import * as Translators from "@mbse/expressions/Translators";
import { Errors, Proxies, Repr, Schemas as S } from "@mbse/schemas/Framework";
import { register } from "@mbse/expressions";

const store = register(new Proxies.OfStore()); // holds every dialect's meta-schemas

const { AttributeError, KeyError, LookupError, ValueError } = Errors;
import { assert, raises, same, text } from "./support.js";

const [P, M, X, L] = [Python.Expressions, Matlab.Expressions, Excel.Expressions, Latex.Expressions];
const [PV, MV, XV] = [Python.Evaluators, Matlab.Evaluators, Excel.Evaluators];
const [C, CV] = [Ccpp.Expressions, Ccpp.Evaluators];
const [SV, SVV] = [SystemVerilog.Expressions, SystemVerilog.Evaluators];
const [BASIC, PYTHON, MATLAB, EXCEL, LATEX] = [E.DIALECT, P.DIALECT, M.DIALECT, X.DIALECT, L.DIALECT];
const DIALECTS = [BASIC, PYTHON, MATLAB, EXCEL, LATEX];
const EVALUATE = new Map<F.Declared, (e: unknown, scope?: any) => unknown>([
  [BASIC, V.OfAny as any], [PYTHON, PV.OfAny], [MATLAB, MV.OfAny], [EXCEL, XV.OfAny]]);
const NUMPY = Translators.Basic_Python.NUMPY;
const self = E.variable("this");
const age = self.age;
const core = age.ge(18n).and_(age.lt(65n).or_(self.has("email").not_())).and_(self.name.ne("x")).or_(
  age.add(1n).sub(2n).mul(3n).neg().gt(0n).and_(age.eq(1n).not_())).and_(self.has("name").eq(true)).and_(
  E.literal(1.5).le(2.5)); // every translatable operation and literal type, operands of one type, no let
const rule = E.let_("a", age, E.variable("a").ge(18n).implies(self.has("email"))); // a let and implies
const permutations = (items: F.Declared[], n: number): F.Declared[][] =>
  n === 0 ? [[]] : items.flatMap((item) => permutations(items.filter((i) => i !== item), n - 1).map((rest) => [item, ...rest]));
const kindOf = (dialect: F.Declared, tag: string) => dialect.kinds().get(tag) as any;
const json = (value: unknown) => globalThis.JSON.stringify(value);
/** The translation of `core` (or another Basic expression) into `dialect`. */
const into = (dialect: F.Declared, expression: any = core) =>
  (dialect === BASIC ? E.OfAny.resolve(expression) : Translators.between(BASIC, dialect).forward(expression));

const Person = new S.OfObject.Builder().ref().properties(text("name"), text("age", BigInt), text("email")).create();
store.register("Person", Person);
const B = (store as any).Person;
const people = [B().name("Ann").age(30n).email("a@x").create(), B().name("Bob").age(70n).create(),
  B().name("x").age(10n).email("c@x").create()];
const [PText, MText, XText, LText, CText, SVText] = [Python.Text, Matlab.Text, Excel.Text, Latex.Text, Ccpp.Text, SystemVerilog.Text];

## TRN-01 · Rules pair patterns; holes bind arguments and attributes

In [ ]:
{
  const [A, B_] = holes("A", "B") as [Hole, Hole];
  const K = new Hole("K", String);
  const double = new Pattern("operation", { name: "add" }, A, A);
  assert(double.size() === 2 && String(double) === "operation(A, A, name='add')" && json([...double.holes()]) === json([["A", "argument"]]));
  const has = new Pattern("operation", { name: "has" }, A, new Pattern("literal", { value: K }));
  assert(has.size() === 3 && json([...has.holes()]) === json([["A", "argument"], ["K", "attribute"]]));
  raises(ValueError, () => new Rule(new Pattern("operation", { name: "f" }, A), new Pattern("operation", { name: "f" }, A, B_)),
    "operation(A, B, name='f') needs the argument hole B, which operation(A, name='f') does not bind");
  raises(ValueError, () => new Rule(new Pattern("operation", { name: "f" }, A), new Pattern("variable", { name: A })),
    "variable(name=A) needs the attribute hole A");
  new Rule(new Pattern("operation", { name: "f" }, A), new Pattern("operation", { name: "f" }, A, B_), "backward"); // only the other way needs B
  raises(ValueError, () => new Rule(double, double, "up"), "direction must be 'both', 'forward' or 'backward', got 'up'");
  for (const declare of [() => new Inline("let", "up"), () => new Elide("import", "up"), () => new Prelude(new Pattern("import", {}, A), "up")]) {
    raises(ValueError, declare, "side must be 'left' or 'right', got 'up'");
  }
  for (const pattern of [new Pattern("import", { module: "numpy" }), new Pattern("import", {}, new Pattern("name", { name: "x" })),
    new Pattern("import", { module: new Hole("M") }, A)]) {
    raises(ValueError, () => new Prelude(pattern, "right"), "a prelude is an import whose one argument is a hole");
  }
  assert(json(renames("operation", "name", "call", "function", { eq: "equal" }, 2).map((r) =>
    [Object.fromEntries(r.left.attributes), Object.fromEntries(r.right.attributes)])) === json([[{ name: "eq" }, { function: "equal" }]]));
  // A conversion translates literals by functions of their attributes, which give null where there is no counterpart.
  const doubleInts = new Convert("literal", "literal", (a) => (typeof a.value === "bigint" ? { value: a.value * 2n } : null),
    (a) => ({ value: (a.value as bigint) / 2n }));
  const doubling = new Pairwise(BASIC, BASIC, [doubleInts]);
  assert(doubling.forward(E.literal(3n)).value === 6n && doubling.backward(E.literal(6n)).value === 3n);
  assert(doubling.inverse().forward(E.literal(6n)).value === 3n && doubling.inverse().backward(E.literal(3n)).value === 6n);
  raises(ValueError, () => doubling.forward(E.literal("a")), "Basic literal 'a' has no Basic counterpart");
  for (const [left, right, named] of [["operation", "literal", "Basic 'operation'"], ["literal", "nothing", "Basic 'nothing'"]]) {
    raises(ValueError, () => new Pairwise(BASIC, BASIC, [new Convert(left as string, right as string, doubleInts.forward, doubleInts.backward)]),
      `a conversion translates literals, and ${named} is not one`);
  }

  // A translator within one dialect is a rewriting: holes bound twice must bind the same term, or equal values.
  const simplify = new Pairwise(BASIC, BASIC, [
    new Rule(double, new Pattern("operation", { name: "mul" }, A, new Pattern("literal", { value: 2n })), "forward"),
    new Rule(new Pattern("operation", { name: "eq" }, new Pattern("literal", { value: K }), new Pattern("literal", { value: K })),
      new Pattern("literal", { value: true }), "forward"),
    new Rule(new Pattern("operation", { name: "add" }, A, B_), new Pattern("operation", { name: "add" }, A, B_), "forward"),
    new Rule(new Pattern("operation", { name: "eq" }, new Pattern("literal", { value: new Hole("V") }), new Pattern("literal", { value: new Hole("W") })),
      new Pattern("literal", { value: false }), "forward"),
    new Rule(new Pattern("variable", { name: new Hole("N") }), new Pattern("variable", { name: new Hole("N") })),
    new Rule(new Pattern("literal", { value: new Hole("V") }), new Pattern("literal", { value: new Hole("V") }), "forward"),
  ]);
  const [x, y] = [E.variable("x"), E.variable("y")];
  assert(F.same(simplify.forward(x.add(x)), x.mul(2n).data) && F.same(simplify.forward(x.add(y)), x.add(y).data));
  assert(simplify.forward(E.literal("a").eq("a")).value === true && simplify.forward(E.literal("a").eq("b")).value === false);
  assert(simplify.forward(E.literal(1n).eq(1n)).value === false); // K holds only a str
  assert(String(simplify) === "<translator Basic <-> Basic>");
}

## TRN-02 · Every pair translates both ways; forms round-trip where both dialects have them

In [ ]:
{
  assert(Translators.TRANSLATORS.length === 12); // every pair of the five, and Basic with Ccpp and SystemVerilog
  for (const [source, target] of permutations(DIALECTS, 2) as [F.Declared, F.Declared][]) {
    const translator = Translators.between(source, target);
    assert(translator.left() === source && translator.right() === target);
    const start = into(source);
    const there = translator.forward(start);
    assert(there.dialect() === target && there.validate({ bound: ["this"], core: true }).length === 0, `${source.name()} ${target.name()}`);
    assert(F.same(translator.backward(there), start), `${source.name()} ${target.name()}`); // round trip
    assert(F.same(translator.inverse().backward(start), there));
  }
  raises(LookupError, () => Translators.between(BASIC, BASIC), "no translator between Basic and Basic");

  assert(PText.ToText(Translators.between(BASIC, PYTHON).forward(rule)) === "(lambda a: hasattr(this, 'email') if a >= 18 else True)(this.age)");
  assert(MText.ToText(Translators.between(BASIC, MATLAB).forward(rule)) === '~(this.age >= 18) || isfield(this, "email")');
  assert(XText.ToText(Translators.between(BASIC, EXCEL).forward(rule)) === "=LET(a, this.age, IF(a >= 18, NOT(ISERROR(this.email)), TRUE))");
  assert(LText.ToText(Translators.between(BASIC, LATEX).forward(rule))
    === "a \\geq 18 \\implies \\operatorname{has}(\\mathit{this}, \\text{email}) \\quad \\text{where } a = \\mathit{this}.\\mathit{age}");
  assert(LText.ToText(Translators.between(PYTHON, LATEX).forward(P.binop("/", P.name("x"), 2n))) === "\\frac{x}{2}");
  // Python, Excel and Latex have lets and implication, so the rule round-trips; Matlab inlines the let and writes
  // implies with ||.
  for (const other of [PYTHON, EXCEL, LATEX]) {
    assert(F.same(Translators.between(other, BASIC).forward(Translators.between(BASIC, other).forward(rule)), rule.data));
  }
  const back = Translators.between(MATLAB, BASIC).forward(Translators.between(BASIC, MATLAB).forward(rule));
  assert(F.same(back, age.ge(18n).not_().or_(self.has("email")).data));

  // NumPy style is a second translator between Basic and Python: numpy's functions, over columns, importing numpy.
  const styled = NUMPY.forward(rule);
  assert(PText.ToText(styled) === "import numpy as np\n(lambda a: np.where(np.greater_equal(a, 18), "
    + "np.logical_not(np.ma.getmaskarray(this['email'])), True))(this['age'])");
  assert(F.same(NUMPY.backward(styled), rule.data) && F.same(NUMPY.backward(NUMPY.forward(core)), core.data));
  assert(PText.ToText(NUMPY.forward(E.literal(1n).add(2n))) === "import numpy as np\nnp.add(1, 2)");
  assert(PText.ToText(NUMPY.forward(E.literal(1n))) === "1" && PText.ToText(NUMPY.forward(self)) === "this"); // no np, no import
  assert(F.same(NUMPY.inverse().forward(styled), rule.data) && NUMPY.inverse().backward(E.literal(1n)).value === 1n);
  const imported = P.import_("math", P.importfrom("numpy", "pi", P.compare(">=", P.attribute(P.name("this"), "age"), 18n)));
  // imports the other side cannot declare are dropped
  for (const target of [BASIC, MATLAB, EXCEL]) {
    assert(F.same(Translators.between(PYTHON, target).forward(imported), into(target, age.ge(18n))));
  }
  assert(XText.ToText(Translators.between(MATLAB, EXCEL).forward(M.import_("pkg.*", M.binary(">", 1n, 0n)))) === "=1 > 0");
  assert(F.same(Translators.between(MATLAB, BASIC).forward(M.import_("pkg.*", M.constant(1n))), E.literal(1n).data));
  raises(ValueError, () => Translators.between(PYTHON, MATLAB).forward(new (kindOf(PYTHON, "import"))("math")), "an import needs a body");
  raises(ValueError, () => Translators.between(PYTHON, BASIC).forward(P.import_("numpy", P.call("np.add", 1n, 2n), "np")),
    "Python call has no Basic counterpart"); // read by NUMPY
}

## TRN-03 · Translating the same pair of dialects by any route gives the same expression

In [ ]:
for (const [a, b, c] of permutations(DIALECTS, 3) as [F.Declared, F.Declared, F.Declared][]) {
  const start = into(a);
  const direct = Translators.between(a, c).forward(start);
  const routed = Translators.between(b, c).forward(Translators.between(a, b).forward(start));
  assert(F.same(direct, routed), `${a.name()} ${b.name()} ${c.name()}`);
}

## TRN-04 · Co-traversal translates each term once, keeps sharing, and traces the correspondence

In [ ]:
{
  const shared = E.operation("add", age, age).data;
  let trace: [any, any][] = [];
  const out = Translators.between(BASIC, EXCEL).forward(shared, trace);
  assert(out.left === out.right); // the shared argument is translated once
  assert(json(trace.map(([s, t]) => [s.kind().KIND, t.kind().KIND])) === json([["variable", "name"], ["operation", "field"],
    ["operation", "infix"]]));
  assert(trace[2]?.[0] === shared && trace[2]?.[1] === out && trace[1]?.[1] === out.left);
  const a = E.variable("a");
  trace = [];
  const inlined = Translators.between(BASIC, MATLAB).forward(E.let_("a", age, a.add(a)), trace);
  assert(MText.ToText(inlined) === "this.age + this.age" && inlined.left === inlined.right);
  assert(same(trace.map(([s]) => s.kind().KIND), ["variable", "operation", "variable", "operation", "let"]));
  const shadowed = E.let_("a", 1n, E.variable("a").add(E.let_("a", 2n, E.variable("a")))); // the innermost binding wins
  assert(MText.ToText(Translators.between(BASIC, MATLAB).forward(shadowed)) === "1 + 2");
  assert(MV.OfAny(Translators.between(BASIC, MATLAB).forward(shadowed)) === 3 && V.OfAny(shadowed) === 3n);
  assert(MText.ToText(Translators.between(EXCEL, MATLAB).forward(X.let_("n", 2n, X.infix("*", X.name("n"), X.name("n"))))) === "2 .* 2");
  const loop = new E.OfOperation.Builder().name("not").create();
  new E.OfOperation.Builder(loop).arguments(loop).update();
  raises(ValueError, () => Translators.between(BASIC, MATLAB).forward(loop), "the expression contains a cycle");
  raises(TypeError, () => Translators.between(BASIC, MATLAB).forward(new E.OfOperation.Data("not", [7n])), "not an expression: 7");
  raises(ValueError, () => Translators.between(BASIC, MATLAB).forward(new E.OfLet.Data("a")), "a let needs a value");
  raises(ValueError, () => Translators.between(EXCEL, MATLAB).forward(new (kindOf(EXCEL, "let"))("a", X.constant(1n))), "a let needs a body");
}

## TRN-05 · What a dialect cannot say is refused, naming the term

In [ ]:
for (const [translate, expression, message] of [
  [(e: any) => Translators.between(BASIC, EXCEL).forward(e), E.literal(new Uint8Array([0])), "Basic literal b'\\x00' has no Excel counterpart"],
  [(e: any) => Translators.between(BASIC, PYTHON).forward(e), new E.OfLiteral.Data(), "Basic literal None has no Python counterpart"],
  [(e: any) => Translators.between(BASIC, MATLAB).forward(e), E.operation("entries", self, "ports"), "Basic operation 'entries' has no Matlab counterpart"],
  [(e: any) => Translators.between(BASIC, PYTHON).forward(e), self.get(E.variable("key")), "Basic operation 'get' has no Python counterpart"],
  [(e: any) => NUMPY.forward(e), new E.OfOperation.Data("has", [self.data, 7n]), "Basic operation 'has'"],
  [(e: any) => Translators.between(BASIC, PYTHON).forward(e), E.operation("eq", 1n, 2n, 3n), "Basic operation 'eq' has no Python counterpart"],
  [(e: any) => Translators.between(EXCEL, BASIC).forward(e), X.function("ISERROR", 1n), "Excel function 'ISERROR' has no Basic counterpart"],
  [(e: any) => Translators.between(EXCEL, BASIC).forward(e), X.function("IF", true, 1n, 2n), "Excel function 'IF' has no Basic counterpart"],
  [(e: any) => Translators.between(EXCEL, BASIC).forward(e), new (kindOf(EXCEL, "let"))("a", X.constant(1n)), "Excel let has no Basic counterpart"],
  [(e: any) => Translators.between(PYTHON, MATLAB).forward(e), P.ifexp(true, 1n, 2n), "Python ifexp has no Matlab counterpart"],
  [(e: any) => Translators.between(EXCEL, MATLAB).forward(e), X.cell("A1"), "Excel cell has no Matlab counterpart"],
  [(e: any) => Translators.between(BASIC, LATEX).forward(e), E.literal(new Uint8Array([0])), "Basic literal b'\\x00' has no Latex counterpart"],
  [(e: any) => Translators.between(LATEX, BASIC).forward(e), L.frac(1n, 2n), "Latex frac has no Basic counterpart"],
  [(e: any) => Translators.between(PYTHON, LATEX).forward(e), P.ifexp(true, 1n, 2n), "Python ifexp has no Latex counterpart"],
  [(e: any) => Translators.between(MATLAB, BASIC).forward(e), M.call("isfield", M.identifier("s"), M.identifier("k")), "Matlab call 'isfield'"],
] as [(e: unknown) => unknown, unknown, string][]) {
  raises(ValueError, () => translate(expression), message);
}

## TRN-06 · Translations evaluate alike where the dialects' rules agree, and apart where they do not

In [ ]:
{
  const truthy = (value: unknown) => PV.truthy(value);
  const EVALUATED = DIALECTS.filter((d) => EVALUATE.has(d)); // Latex has no evaluator
  for (const person of people) {
    for (const expression of [core, rule]) {
      const values = EVALUATED.map((d) => truthy(EVALUATE.get(d)?.(into(d, expression), { this: person })));
      assert(new Set(values).size === 1, `${person.name} ${values}`);
    }
  }
  // NumPy evaluates all three at once, over columns: the Python implementation's only.

  const nobody = B().name("Nobody").create(); // no age: each dialect's own notion of a missing value
  const missing = age.ge(18n);
  assert(V.OfAny(missing, { this: nobody }) === null); // Basic: unknown
  raises(AttributeError, () => PV.OfAny(into(PYTHON, missing), { this: nobody }), "'Person' object has no attribute 'age'"); // Python raises
  assert(XV.OfAny(into(EXCEL, missing), { this: nobody }) === XV.Error.of("#FIELD!")); // an error value
  raises(KeyError, () => MV.OfAny(into(MATLAB, missing), { this: nobody }), 'Unrecognized field name "age".'); // MATLAB raises
  const [mixed, cased] = [E.literal(1n).eq(1.0), E.literal("a").eq("A")];
  assert(V.OfAny(mixed) === null && V.OfAny(cased) === false); // Basic never coerces, and compares text exactly
  assert(same(EVALUATED.slice(1).map((d) => truthy(EVALUATE.get(d)?.(into(d, mixed)))), [true, true, true]));
  assert(same(EVALUATED.slice(1).map((d) => truthy(EVALUATE.get(d)?.(into(d, cased)))), [false, false, true]));
}

## TRN-07 · The bitwise operations translate between Basic, Python, MATLAB and Excel, and evaluate alike on non-negative integers

In [ ]:
{
  const x = E.variable("x");
  const bits = x.bitand(12n).bitor(x.shr(2n)).bitxor(x.shl(1n));
  const RENDER = new Map<F.Declared, (e: any) => string>([[PYTHON, PText.ToText], [MATLAB, MText.ToText], [EXCEL, XText.ToText]]);
  const expected = new Map<F.Declared, string>([[PYTHON, "(x & 12 | x >> 2) ^ x << 1"],
    [MATLAB, "bitxor(bitor(bitand(x, 12), bitshift(x, -2)), bitshift(x, 1))"],
    [EXCEL, "=BITXOR(BITOR(BITAND(x, 12), BITRSHIFT(x, 2)), BITLSHIFT(x, 1))"]]);
  for (const [dialect, source] of expected) {
    const translator = Translators.between(BASIC, dialect);
    const translated = translator.forward(bits);
    assert(RENDER.get(dialect)!(translated) === source && F.same(translator.inverse().forward(translated), bits.data));
    assert(Number(EVALUATE.get(dialect)!(translated, { x: 13n })) === 21 && V.OfAny(bits, { x: 13n }) === 21n);
  }
  // Between the other dialects too, by any route.
  for (const [a, b] of [[PYTHON, MATLAB], [PYTHON, EXCEL], [MATLAB, EXCEL]] as [F.Declared, F.Declared][]) {
    const source = Translators.between(BASIC, a).forward(bits);
    const target = Translators.between(a, b).forward(source);
    assert(F.same(target, Translators.between(BASIC, b).forward(bits)) && F.same(Translators.between(b, a).forward(target), source));
  }
  // bitnot is Python's ~ and NumPy's invert; MATLAB and Excel have none for their numbers.
  assert(PText.ToText(Translators.between(BASIC, PYTHON).forward(x.bitnot())) === "~x");
  assert(PText.ToText(NUMPY.forward(x.bitnot().shl(1n))) === "import numpy as np\nnp.left_shift(np.invert(x), 1)");
  raises(ValueError, () => Translators.between(BASIC, MATLAB).forward(x.bitnot()), "Basic operation 'bitnot' has no Matlab counterpart");
  raises(ValueError, () => Translators.between(BASIC, EXCEL).forward(x.bitnot()), "Basic operation 'bitnot' has no Excel counterpart");
  raises(ValueError, () => Translators.between(PYTHON, MATLAB).forward(P.unaryop("~", P.name("x"))),
    "Python unaryop '~' has no Matlab counterpart");
  // Basic's and Python's integers are unbounded and signed; MATLAB's and Excel's bit functions are not.
  const mask = x.bitand(1n);
  assert(V.OfAny(mask, { x: -1n }) === 1n && PV.OfAny(Translators.between(BASIC, PYTHON).forward(mask), { x: -1n }) === 1n);
  assert(XV.OfAny(Translators.between(BASIC, EXCEL).forward(mask), { x: -1n }) === XV.Error.of("#NUM!"));
}

## TRN-08 · Basic and Ccpp translate into each other; lets are inlined, and `has`, `->` and typed constants have no counterpart

In [ ]:
{
  const CCPP = C.DIALECT;
  const [x, me] = [E.variable("x"), E.variable("this")];
  const rule = E.let_("a", me.age, E.variable("a").ge(18n).and_(me.width.shl(2n).bitand(255n).ne(0n)).implies(me.name.eq("x")));
  const translator = Translators.between(BASIC, CCPP);
  const translated = translator.forward(rule);
  assert(CText.ToText(translated) === '!(this.age >= 18 && (this.width << 2 & 255) != 0) || this.name == "x"');
  const expected = E.operation("or", E.operation("not", me.age.ge(18n).and_(me.width.shl(2n).bitand(255n).ne(0n))), me.name.eq("x"));
  assert(F.same(translator.inverse().forward(translated), expected.data));
  const every = x.add(1n).sub(2n).mul(3n).neg().bitor(x.bitxor(1n)).bitand(x.bitnot()).shr(1n).lt(x).or_(x.le(1n).and_(x.gt(2n).not_())).ne(
    x.ge(0n).eq(E.literal(true)));
  assert(F.same(translator.inverse().forward(translator.forward(every)), every.data) && CText.ToText(translator.forward(E.literal(1.5))) === "1.5");
  for (const value of [3n, 9n]) { // C's ints and Basic's agree where neither overflows
    assert(CV.OfAny(translator.forward(x.add(1n).mul(2n).gt(5n).and_(x.lt(9n))), { x: value }) === V.OfAny(x.add(1n).mul(2n).gt(5n).and_(x.lt(9n)), { x: value }));
  }
  // Typed constants are literals of value domains: C's integer types by width and signedness, its floating types by format.
  const U8 = new D.OfInteger.Data(8n, false);
  const typedLiterals: [any, string][] = [[E.literal(200n, U8), "(uint8_t)200"], [E.literal(-5n, new D.OfInteger.Data(64n)), "(int64_t)-5"],
    [E.literal(1.5, new D.OfIeee754.Data("binary32")), "1.5f"], [E.literal("0.1", new D.OfIeee754.Data("binary128")), "0.1L"]];
  for (const [literal, source] of typedLiterals) {
    const there = translator.forward(literal);
    assert(CText.ToText(there) === source && F.same(translator.inverse().forward(there), literal.data), source);
  }
  const typedConstants: [unknown, any][] = [[C.constant(5n, "unsigned int"), new D.OfInteger.Data(32n, false)],
    [C.constant(5n, "size_t"), new D.OfInteger.Data(64n, false)]];
  for (const [constant, domain] of typedConstants) assert(domain.equals(translator.inverse().forward(constant).domain));
  for (const [constant, plain] of [[C.constant(true, "bool"), true], [C.constant(1.5, "double"), 1.5]] as const) { // binary64 is a float's own domain
    assert(F.same(translator.inverse().forward(constant), E.literal(plain).data));
  }
  assert(CV.OfAny(translator.forward(E.literal(200n, U8).add(E.literal(100n, U8)))) === 300n); // in C's int, where Basic's uint8 overflows
  const refused: [unknown, F.Declared, string][] = [
    [me.has("email"), BASIC, "Basic operation 'has' has no Ccpp counterpart"],
    [E.literal(200n, new D.OfInteger.Data(12n)), BASIC, "Basic literal 200 of int12 has no Ccpp counterpart"],
    [E.literal(new Uint8Array([1]), new D.OfBits.Data(8n)), BASIC, "Basic literal b'\\x01' of bits8 has no Ccpp counterpart"],
    [C.constant(5n, "word"), CCPP, "Ccpp constant 5 has no Basic counterpart"],
    [C.member(C.identifier("x"), "a", "->"), CCPP, "Ccpp member has no Basic counterpart"],
  ];
  for (const [expression, source, message] of refused) {
    raises(ValueError, () => Translators.between(source, source === CCPP ? BASIC : CCPP).forward(expression), message);
  }
}

## TRN-09 · Basic and SystemVerilog translate into each other; lets are inlined, bools are single bits, and `has`, sized vectors and 4-state comparisons have no counterpart

In [ ]:
{
  const SYSTEMVERILOG = SV.DIALECT;
  const [x, me] = [E.variable("x"), E.variable("this")];
  const rule = E.let_("a", me.age, E.variable("a").ge(18n).and_(me.width.shr(2n).bitand(255n).ne(0n)).implies(me.name.eq("x")));
  const translator = Translators.between(BASIC, SYSTEMVERILOG);
  const translated = translator.forward(rule);
  assert(SVText.ToText(translated) === 'this.age >= 18 && (this.width >>> 2 & 255) != 0 -> this.name == "x"');
  const expected = me.age.ge(18n).and_(me.width.shr(2n).bitand(255n).ne(0n)).implies(me.name.eq("x"));
  assert(F.same(translator.inverse().forward(translated), expected.data));
  const every = x.add(1n).sub(2n).mul(3n).neg().bitor(x.bitxor(1n)).bitand(x.bitnot()).shl(1n).lt(x).or_(x.le(1n).and_(x.gt(2n).not_())).ne(
    x.ge(0n).eq(E.literal(true)).or_(E.literal(false)));
  assert(F.same(translator.inverse().forward(translator.forward(every)), every.data));
  assert(SVText.ToText(translator.forward(E.literal(true).and_(E.literal(1.5).lt(2n)))) === "1'b1 && 1.5 < 2");
  for (const value of [3n, 9n]) { // SystemVerilog's integers and Basic's agree where neither overflows
    const check = x.add(1n).mul(2n).gt(5n).and_(x.lt(9n));
    assert((SVV.OfAny(translator.forward(check), { x: value }) as any).integer() === (V.OfAny(check, { x: value }) ? 1n : 0n));
  }
  // Sized vectors are literals of value domains, by their base: decimal integers, hexadecimal bits, binary std_logic.
  const [U8, LOGIC] = [new D.OfInteger.Data(8n, false), new D.OfIeee1164.Data()];
  const typedLiterals: [any, string][] = [[E.literal(200n, U8), "8'd200"], [E.literal(-5n, new D.OfInteger.Data(8n)), "8'sd251"],
    [E.literal(5n, new D.OfInteger.Data(12n)), "12'sd5"], [E.literal(new Uint8Array([0x0a, 0xbc]), new D.OfBits.Data(12n)), "12'habc"],
    [E.literal("X", LOGIC), "1'bx"], [E.literal("1", LOGIC), "1'b1"],
    [E.literal("10XZ", new D.OfIeee1164.Data(4n)), "4'b10xz"], [E.literal("1010", new D.OfIeee1164.Data(4n)), "4'b1010"]];
  for (const [literal, source] of typedLiterals) {
    const there = translator.forward(literal);
    assert(SVText.ToText(there) === source && F.same(translator.inverse().forward(there), literal.data), source);
  }
  const vectors: [unknown, unknown, any][] = [[SV.vector("1010"), new Uint8Array([0x0a]), new D.OfBits.Data(4n)],
    [SV.vector("1111", true, "h"), -1n, new D.OfInteger.Data(4n)], [SV.vector("101", null, "o"), new Uint8Array([5]), new D.OfBits.Data(3n)],
    [SV.vector("z"), "Z", LOGIC], [SV.vector("0", null, "b"), "0", LOGIC],
    [SV.vector("10x1"), "10X1", new D.OfIeee1164.Data(4n)], [SV.vector("x1", true, "h"), "X1", new D.OfIeee1164.Data(2n)],
    [SV.vector("1010", null, "b"), "1010", new D.OfIeee1164.Data(4n)], [SV.vector("1", true, "b"), -1n, new D.OfInteger.Data(1n)]];
  for (const [vector, value, domain] of vectors) {
    const literal = translator.inverse().forward(vector);
    assert(json(Repr.repr(literal.value)) === json(Repr.repr(value)) && domain.equals(literal.domain), SVText.ToText(vector));
  }
  assert(String(SVV.OfAny(translator.forward(E.literal(200n, U8).add(E.literal(100n, U8))))) === "8'b00101100"); // wraps in 8 bits, where Basic's overflows
  const refused: [unknown, F.Declared, string][] = [
    [me.has("email"), BASIC, "Basic operation 'has' has no SystemVerilog counterpart"],
    [E.literal(1.5, new D.OfIeee754.Data("binary32")), BASIC, "Basic literal 1.5 of binary32 has no SystemVerilog counterpart"],
    [E.literal("U", new D.OfIeee1164.Data()), BASIC, "Basic literal 'U' of std_logic has no SystemVerilog counterpart"],
    [E.literal("1U", new D.OfIeee1164.Data(2n)), BASIC, "Basic literal '1U' of std_logic_vector2 has no SystemVerilog counterpart"],
    [SV.binary("===", SV.identifier("x"), 1n), SYSTEMVERILOG, "SystemVerilog binary '===' has no Basic counterpart"],
  ];
  for (const [expression, source, message] of refused) {
    raises(ValueError, () => Translators.between(source, source === SYSTEMVERILOG ? BASIC : SYSTEMVERILOG).forward(expression), message);
  }
}

## TRN-10 · Collections translate to Python's builtins and generator expressions, and evaluate alike over lists

In [ ]:
{
  const PY = Translators.between(BASIC, PYTHON);
  const [me, p] = [E.variable("this"), E.variable("p")];
  const rule = me.pins.all("p", p.gt(0n)).and_(me.pins.any("p", p.in_(me.allowed))).and_(
    me.pins.count_where("p", p.ge(2n)).le(me.pins.count())).and_(
    me.pins.item(0n).add(me.pins.sum()).lt(me.pins.max().mul(me.pins.min()).add(100n)));
  const there = PY.forward(rule);
  assert(PText.ToText(there) === "all(p > 0 for p in this.pins) and any(p in this.allowed for p in this.pins) and "
    + "sum(1 for p in this.pins if p >= 2) <= len(this.pins) and "
    + "this.pins[0] + sum(this.pins) < max(this.pins) * min(this.pins) + 100");
  assert(F.same(PY.backward(there), rule.data) && F.same(PY.inverse().forward(there), rule.data));
  const unique = me.pins.unique(); // written as an idiom, which does not read back
  assert(PText.ToText(PY.forward(unique)) === "len(set(this.pins)) == len(this.pins)");
  const Bus = new S.OfObject.Builder().ref().properties(
    (q) => q.name("pins").of((t) => t.as_indexed((i) => i.of(new S.OfNative.Data(BigInt)))),
    (q) => q.name("allowed").of((t) => t.as_indexed((i) => i.of(new S.OfNative.Data(BigInt))))).create();
  store.register("Bus", Bus);
  for (const [pins, allowed] of [[[1n, 2n, 3n], [3n]], [[1n, 1n], [5n]], [[0n, 2n], [2n]]]) {
    const bus = (store as any).Bus().pins(pins).allowed(allowed).create();
    for (const expression of [rule, unique]) {
      assert(V.OfAny(expression, { this: bus }) === PV.OfAny(PY.forward(expression), { this: bus }), json(String(pins)));
    }
  }
  const refused: [unknown, any, string][] = [
    [me.entries("ports"), PY, "Basic operation 'entries' has no Python counterpart"],
    [me.pins.count_where("p", p.gt(0n)), NUMPY, "Basic quantifier 'count' has no Python counterpart"],
    [P.call("all", P.name("xs")), PY.inverse(), "Python call has no Basic counterpart"],
    [P.call("sum", P.generator("p", P.name("xs"), P.name("p"))), PY.inverse(), "Python generator has no Basic counterpart"],
    [PY.forward(unique), PY.inverse(), "Python call has no Basic counterpart"],
  ];
  for (const [expression, translator, message] of refused) raises(ValueError, () => translator.forward(expression), message);
}

## TRN-11 · Collections translate to SystemVerilog's array methods, and evaluate alike over arrays

In [ ]:
{
  const SVT = Translators.between(BASIC, SV.DIALECT);
  const [me, p] = [E.variable("this"), E.variable("p")];
  const rule = me.pins.all("p", p.gt(0n)).and_(me.pins.any("p", p.in_(me.allowed))).and_(
    me.pins.count_where("p", p.ge(2n)).le(me.pins.count())).and_(
    me.pins.item(0n).add(me.pins.sum()).lt(me.pins.max().mul(me.pins.min()).add(100n))); // TRN-10's
  const unique = me.pins.unique();
  const there = SVT.forward(rule);
  assert(SVText.ToText(there) === "this.pins.and(p) with (p > 0) && this.pins.or(p) with (p inside {this.allowed}) && "
    + "this.pins.sum(p) with (int'(p >= 2)) <= this.pins.size() && "
    + "this.pins[0] + this.pins.sum() < this.pins.max()[0] * this.pins.min()[0] + 100");
  assert(F.same(SVT.backward(there), rule.data));
  assert(SVText.ToText(SVT.forward(unique)) === "this.pins.unique().size() == this.pins.size()"); // an idiom, which does not read back
  for (const [pins, allowed] of [[[1n, 2n, 3n], [3n]], [[1n, 1n], [5n]], [[0n, 2n], [2n]]]) {
    const bus = (store as any).Bus().pins(pins).allowed(allowed).create();
    for (const expression of [rule, unique]) {
      assert((SVV.OfAny(SVT.forward(expression), { this: bus }) as any).integer() === (V.OfAny(expression, { this: bus }) ? 1n : 0n), String(pins));
    }
  }
  const xs = SV.identifier("xs");
  const refused: [unknown, any, string][] = [
    [me.entries("ports"), SVT, "Basic operation 'entries' has no SystemVerilog counterpart"],
    [SV.method(xs, "product"), SVT.inverse(), "SystemVerilog method 'product' has no Basic counterpart"],
    [SV.iterate(xs, "xor", "p", SV.identifier("p")), SVT.inverse(), "SystemVerilog iterate 'xor' has no Basic counterpart"],
    [SVT.forward(unique), SVT.inverse(), "SystemVerilog method 'unique' has no Basic counterpart"],
  ];
  for (const [expression, translator, message] of refused) raises(ValueError, () => translator.forward(expression), message);
}

## TRN-12 · Collections translate to MATLAB's arrays: arrayfun, indexing from 1 and array functions, and evaluate alike

In [ ]:
{
  const MT = Translators.between(BASIC, MATLAB);
  const [me, p] = [E.variable("this"), E.variable("p")];
  const rule = me.pins.all("p", p.gt(0n)).and_(me.pins.any("p", p.in_(me.allowed))).and_(
    me.pins.count_where("p", p.ge(2n)).le(me.pins.count())).and_(
    me.pins.item(0n).add(me.pins.sum()).lt(me.pins.max().mul(me.pins.min()).add(100n))); // TRN-10's
  const unique = me.pins.unique();
  const there = MT.forward(rule);
  assert(MText.ToText(there) === "all(arrayfun(@(p) p > 0, this.pins)) && any(arrayfun(@(p) ismember(p, this.allowed), this.pins)) && "
    + "nnz(arrayfun(@(p) p >= 2, this.pins)) <= numel(this.pins) && "
    + "this.pins(0 + 1) + sum(this.pins) < max(this.pins) .* min(this.pins) + 100");
  assert(F.same(MT.backward(there), rule.data));
  assert(MText.ToText(MT.forward(unique)) === "numel(unique(this.pins)) == numel(this.pins)"); // an idiom, which does not read back
  for (const [pins, allowed] of [[[1n, 2n, 3n], [3n]], [[1n, 1n], [5n]], [[0n, 2n], [2n]]]) {
    const bus = (store as any).Bus().pins(pins).allowed(allowed).create();
    for (const expression of [rule, unique]) {
      assert(MV.OfAny(MT.forward(expression), { this: bus }) === V.OfAny(expression, { this: bus }), String(pins));
    }
  }
  const xs = M.identifier("xs");
  const refused: [unknown, any, string][] = [
    [me.entries("ports"), MT, "Basic operation 'entries' has no Matlab counterpart"],
    [M.index(xs, 1n), MT.inverse(), "Matlab index has no Basic counterpart"], // only xs(i + 1) reads back
    [M.call("all", xs), MT.inverse(), "Matlab call 'all' has no Basic counterpart"],
    [MT.forward(unique), MT.inverse(), "Matlab call 'unique' has no Basic counterpart"],
  ];
  for (const [expression, translator, message] of refused) raises(ValueError, () => translator.forward(expression), message);
}

## TRN-13 · Collections translate to Excel's arrays: MAP and LAMBDA, INDEX and MATCH and array functions, and evaluate alike

In [ ]:
{
  const XT = Translators.between(BASIC, EXCEL);
  const [me, p] = [E.variable("this"), E.variable("p")];
  const rule = me.pins.all("p", p.gt(0n)).and_(me.pins.any("p", p.in_(me.allowed))).and_(
    me.pins.count_where("p", p.ge(2n)).le(me.pins.count())).and_(
    me.pins.item(0n).add(me.pins.sum()).lt(me.pins.max().mul(me.pins.min()).add(100n))); // TRN-10's
  const unique = me.pins.unique();
  const there = XT.forward(rule);
  assert(XText.ToText(there) === "=AND(AND(AND(AND(MAP(this.pins, LAMBDA(p, p > 0))), OR(MAP(this.pins, LAMBDA(p, ISNUMBER(MATCH(p, this.allowed, 0)))))), "
    + "SUM(MAP(this.pins, LAMBDA(p, IF(p >= 2, 1, 0)))) <= ROWS(this.pins)), "
    + "INDEX(this.pins, 0 + 1) + SUM(this.pins) < MAX(this.pins) * MIN(this.pins) + 100)");
  assert(F.same(XT.backward(there), rule.data));
  assert(XText.ToText(XT.forward(unique)) === "=ROWS(UNIQUE(this.pins)) = ROWS(this.pins)"); // an idiom, which does not read back
  for (const [pins, allowed] of [[[1n, 2n, 3n], [3n]], [[1n, 1n], [5n]], [[0n, 2n], [2n]]]) {
    const bus = (store as any).Bus().pins(pins).allowed(allowed).create();
    for (const expression of [rule, unique]) {
      assert(XV.OfAny(XT.forward(expression), { this: bus }) === V.OfAny(expression, { this: bus }), String(pins));
    }
  }
  const xs = X.name("xs");
  const refused: [unknown, any, string][] = [
    [me.entries("ports"), XT, "Basic operation 'entries' has no Excel counterpart"],
    [X.function("INDEX", xs, 1n), XT.inverse(), "Excel function 'INDEX' has no Basic counterpart"], // only INDEX(xs, i + 1) reads back
    [X.function("AND", xs), XT.inverse(), "Excel function 'AND' has no Basic counterpart"],
    [XT.forward(unique), XT.inverse(), "Excel function 'UNIQUE' has no Basic counterpart"],
  ];
  for (const [expression, translator, message] of refused) raises(ValueError, () => translator.forward(expression), message);
}

## TRN-14 · Collections translate directly between Python, MATLAB and Excel, as they do through Basic

In [ ]:
{
  const [me, p] = [E.variable("this"), E.variable("p")];
  const rule = me.pins.all("p", p.gt(0n)).and_(me.pins.any("p", p.in_(me.allowed))).and_(
    me.pins.count_where("p", p.ge(2n)).le(me.pins.count())).and_(
    me.pins.item(0n).add(me.pins.sum()).lt(me.pins.max().mul(me.pins.min()).add(100n))); // TRN-10's
  for (const [left, right] of [[PYTHON, MATLAB], [PYTHON, EXCEL], [MATLAB, EXCEL]] as [F.Declared, F.Declared][]) {
    const source = Translators.between(BASIC, left).forward(rule);
    const there = Translators.between(left, right).forward(source);
    assert(F.same(there, Translators.between(BASIC, right).forward(rule)), `${left.name()} ${right.name()}`);
    assert(F.same(Translators.between(right, left).forward(there), source), `${left.name()} ${right.name()}`);
  }
  raises(ValueError, () => Translators.between(PYTHON, MATLAB).forward(Translators.between(BASIC, PYTHON).forward(me.pins.unique())),
    "Python call has no Matlab counterpart"); // unique's idiom does not translate
}